# Explore the CDK2 target in ChEMBL

**Exploration only.** Nothing here writes to disk and no curation decisions are
made. The point is to look at what data exists before choosing a policy.

Two questions:

1. Which ChEMBL target is *actually* human CDK2?
2. How many bioactivity measurements exist, and of what kind?

Reusable logic lives in `cdk2moo.data`; this notebook only calls it.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path().resolve().parent / "src"))

from cdk2moo import config
from cdk2moo.data import search_targets, match_by_accession, count_activities

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)

## 1. What does a name search return?

Searching `"CDK2"` is fuzzy. Expect to see the CDK2/cyclin A protein complex,
possibly non-human orthologues, and assorted noise alongside the thing we want.

In [2]:
rows = search_targets(config.TARGET_NAME)
print(f"{len(rows)} hits\n")

df = pd.DataFrame(rows)
df["accessions"] = df["accessions"].apply(lambda a: ",".join(a))
df

36 hits



,chembl_id,pref_name,organism,target_type,accessions
0,CHEMBL6193825,von Hippel-Lindau disease tumor suppressor/CDK2,Danio rerio,PROTEIN-PROTEIN INTERACTION,"Q7ZWB1,Q68EH9"
1,CHEMBL3038517,CDK2/CDK4,Homo sapiens,PROTEIN FAMILY,"P11802,P24941"
2,CHEMBL4106185,CDK2/CDK4,Mus musculus,PROTEIN FAMILY,"P30285,P97377"
3,CHEMBL4106186,CDK2/CDK9,Mus musculus,PROTEIN FAMILY,"Q99J95,P97377"
4,CHEMBL3038469,CDK2/Cyclin A2,Homo sapiens,PROTEIN COMPLEX,"P24941,P20248"
5,CHEMBL3038470,CDK2/Cyclin A1,Homo sapiens,PROTEIN COMPLEX,"P24941,P78396"
6,CHEMBL4106152,CDK2/Cyclin O,Homo sapiens,PROTEIN COMPLEX,"P24941,P22674"
7,CHEMBL4523633,CDK2/Cyclin-E2,Homo sapiens,PROTEIN COMPLEX,"P24941,O96020"
8,CHEMBL4523634,CDK2/Cyclin-Y,Homo sapiens,PROTEIN COMPLEX,"P24941,Q8ND76"
9,CHEMBL4888454,CDK2/Cyclin C,Homo sapiens,PROTEIN COMPLEX,"P24941,P24863"


## 2. Identity by accession, not by name

ChEMBL target names are inconsistent; UniProt accessions are not. Human CDK2 is
**P24941** — confirmed independently against the RCSB entry for 4KD1, which is the
structure we will dock into. Matching on the accession means the ligand data and
the crystal structure are provably the same protein.

We also require `target_type == "SINGLE PROTEIN"`, which excludes the CDK2/cyclin A
complex. Activity measured against the complex is a different experiment: cyclin A
binding activates the kinase and changes the pocket, so pooling the two would mix
two conditions under one label.

In [3]:
matches = match_by_accession(
    rows,
    uniprot=config.TARGET_UNIPROT,
    organism=config.TARGET_ORGANISM,
)

for m in matches:
    print(f"{m['chembl_id']}  {m['pref_name']}  ({m['organism']})")

assert len(matches) >= 1, "No single-protein match for P24941 - stop and investigate."
if len(matches) > 1:
    print("\nMore than one match - inspect before continuing.")

target_id = matches[0]["chembl_id"]
target_id

CHEMBL301  Cyclin-dependent kinase 2  (Homo sapiens)


'CHEMBL301'

## 3. How much data is there?

Two filters are applied to get the `clean` column:

- **`standard_relation == "="`** drops censored values such as `IC50 > 10000 nM`.
  Those carry real information (the compound is weak) but cannot be regressed on.
  Note the consequence: what survives is biased toward *actives*. Our surrogate will
  barely have seen a weak compound, which is worth remembering when it later predicts
  high activity for something strange.

- **`standard_units == "nM"`** drops records ChEMBL could not normalise. Usually a
  different kind of measurement or a curation oddity. Cheap insurance against a
  unit-conversion bug that would be invisible in the final numbers.

Each count is a separate API call, so this cell takes a minute or two.

In [4]:
counts = count_activities(target_id, config.MEASUREMENT_TYPES)

print(f"Total activity records of any kind: {counts['total']}\n")

cdf = pd.DataFrame(counts["per_type"]).T
cdf["kept_pct"] = (100 * cdf["clean"] / cdf["all"]).round(1)
cdf.sort_values("clean", ascending=False)

Total activity records of any kind: 10355



,all,clean,kept_pct
IC50,3692,2798,75.8
Ki,1210,611,50.5
Kd,465,119,25.6
AC50,8,8,100.0
EC50,2,2,100.0
Potency,0,0,NaN
Inhibition,2245,0,0.0


## 4. What to decide from this

Record the answers, then set them in `config.py`:

- `TARGET_CHEMBL_ID` — the verified ID from section 2.
- `KEEP_TYPES` — which measurement types to use.

The judgment call is whether to pool IC50, Ki and Kd into one label column.
They measure related but not identical quantities: IC50 depends on the assay's ATP
concentration (the compound competes with ATP for the same pocket, so more ATP makes
the same compound look weaker), while Ki corrects for that and Kd sidesteps it by
measuring binding directly.

Pooling buys sample size and pays for it in label noise. Not pooling is cleaner but
may leave too few compounds. Decide with the numbers above in front of you — and
whichever way it goes, be able to defend it.